本次以英雄联盟对局胜负预测任务为基础，要求实现决策树算法相关细节，加深对算法的理解，并了解做机器学习任务的大致流程。

### 任务介绍
英雄联盟（League of Legends，LoL）是一个多人在线竞技游戏，由拳头游戏（Riot Games）公司出品。在游戏中，每位玩家控制一位有独特技能的英雄，红蓝两支队伍各有五位玩家进行对战，目标是摧毁对方的基地水晶。水晶有多座防御塔保护，通常需要先摧毁一些防御塔再摧毁水晶。玩家所控制的英雄起初非常弱，需要不断击杀小兵、野怪和对方英雄来获得金币、经验。经验可以提升英雄等级和技能等级，金币可以用来购买装备提升攻击、防御等属性。对战过程中一般没有己方单位在附近的地点是没有视野的，即无法看到对面单位，双方可以通过使用守卫来监视某个地点，洞察对面走向、制定战术。
本数据集来自[Kaggle](https://www.kaggle.com/bobbyscience/league-of-legends-diamond-ranked-games-10-min)，包含了9879场钻一到大师段位的单双排对局，对局双方几乎是同一水平。每条数据是前10分钟的对局情况，每支队伍有19个特征，红蓝双方共38个特征。这些特征包括英雄击杀、死亡，金钱、经验、等级情况等等。一局游戏一般会持续30至40分钟，但是实际前10分钟的局面很大程度上影响了之后胜负的走向。作为最成功的电子竞技游戏之一，对局数据、选手数据的量化与研究具有重要意义，可以启发游戏将来的发展和改进。

本任务是希望同学们依据注释的要求，对代码中空缺部分进行填写，**完成决策树模型的详细实现**，根据已有的对局前10分钟特征信息，预测最后获胜方是蓝色方还是红色方，了解执行一个**机器学习任务的大致流程**，并**提交代码和实验报告**。第一次作业也是一个机器学习小实验的例子，之后的作业可能不再提供预处理等流程代码，由同学们自己设计实验完成代码编写。

### 导入工具包
pandas是数据分析和处理常用的工具包，非常适合处理行列表格数据。numpy是数学运算工具包，支持高效的矩阵、向量运算。sklearn是机器学习常用工具包，包括了一些已经实现好的简单模型和一些常用数据处理方法、评价指标等函数。

In [ ]:
import pandas as pd  # 数据处理
import numpy as np  # 数学运算
from sklearn.model_selection import train_test_split  # 划分数据集函数
from sklearn.metrics import accuracy_score  # 准确率函数
from collections import Counter, defaultdict  # 统计函数和字典

RANDOM_SEED = 2020  # 固定随机种子

### 读入数据
假设数据文件放在`./data/`目录下，标准的csv文件可以用pandas里的`read_csv()`函数直接读入。文件共有40列，38个特征（红蓝方各19），1个标签列（blueWins），和一个对局标号（gameId）。对局标号不是标签也不是特征，可以舍去。

In [ ]:
csv_data = "./data/high_diamond_ranked_10min.csv"  # 数据路径
data_df = pd.read_csv(csv_data, sep=",")  # 读入csv文件为pandas的DataFrame
data_df = data_df.drop(columns="gameId")  # 舍去对局标号列

###  数据概览
对于一个机器学习问题，在拿到任务和数据后，首先需要观察数据的情况，比如我们可以通过`.iloc[0]`取出数据的第一行并输出。不难看出每个特征都存成了float64浮点数，该对局蓝色方开局10分钟有小优势。同时也可以发现有些特征列是重复冗余的，比如blueGoldDiff表示蓝色队金币优势，redGoldDiff表示红色方金币优势，这两个特征是完全对称的互为相反数。blueCSPerMin是蓝色方每分钟击杀小兵数，它乘10就是10分钟所有小兵击杀数blueTotalMinionsKilled。在之后的特征处理过程中可以考虑去除这些冗余特征。
另外，pandas有非常方便的`describe()`函数，可以直接通过DataFrame进行调用，可以展示每一列数据的一些统计信息，对数据分布情况有大致了解，比如blueKills蓝色方击杀英雄数在前十分钟的平均数是6.14、方差为2.93，中位数是6，百分之五十以上的对局中该特征在4-8之间，等等。

In [ ]:
print(data_df.iloc[0])  # 输出第一行数据
data_df.describe()  # 每列特征的简单统计信息

### 增删特征
传统的机器学习模型大部分都是基于特征的，因此特征工程是机器学习中非常重要的一步。有时构造一个好的特征比改进一个模型带来的提升更大。这里简单展示一些特征处理的例子。首先，上面提到，特征列中有些特征信息是完全冗余的，会给模型带来不必要的计算量，可以去除。其次，相比于红蓝双方击杀、助攻的绝对值，可能双方击杀英雄的差值更能体现出当前对战的局势。因此，我们可以构造红蓝双方对应特征的差值。数据文件中已有的差值是金币差GoldDiff和经验差ExperienceDiff，实际上每个对应特征都可以构造这样的差值特征。

In [ ]:
drop_features = [
    "blueGoldDiff",
    "redGoldDiff",
    "blueExperienceDiff",
    "redExperienceDiff",
    "blueCSPerMin",
    "redCSPerMin",
    "blueGoldPerMin",
    "redGoldPerMin",
]  # 需要舍去的特征列
df = data_df.drop(columns=drop_features)  # 舍去特征列
info_names = [
    c[3:] for c in df.columns if c.startswith("red")
]  # 取出要作差值的特征名字（除去red前缀）
for info in info_names:  # 对于每个特征名字
    df["br" + info] = (
        df["blue" + info] - df["red" + info]
    )  # 构造一个新的特征，由蓝色特征减去红色特征，前缀为br
# 其中FirstBlood为首次击杀最多有一只队伍能获得，brFirstBlood=1为蓝，0为没有产生，-1为红
df = df.drop(columns=["blueFirstBlood", "redFirstBlood"])  # 原有的FirstBlood可删除

### 特征离散化
决策树ID3算法一般是基于离散特征的，本例中存在很多连续的数值特征，例如队伍金币。直接应用该算法每个值当作一个该特征的一个取值可能造成严重的过拟合，因此需要对特征进行离散化，即将一定范围内的值映射成一个值，例如对用户年龄特征，将0-10映射到0，11-18映射到1，19-25映射到2，25-30映射到3，等等类似，然后在决策树构建时使用映射后的值计算信息增益。

***本小节要求实现特征离散化，请补全相关代码***

In [ ]:
def discretize_features(df, label_col="blueWins", method="quantile", bins=5):
    """
    特征离散化函数，支持连续特征自动分箱

    :param df: 原始数据（DataFrame格式）
    :param label_col: 标签列名（默认为'blueWins'）
    :param method: 分箱方法（'quantile'等频分箱/'uniform'等距分箱，默认为'quantile'）
    :param bins: 分箱数（每个连续特征将被划分为bins个区间，默认为5）
    :return: 离散化后的数据（DataFrame格式），分箱阈值字典（记录各连续特征的分箱边界）
    """

    discrete_df = df.copy()  # 复制一份数据，避免修改原始数据
    feature_cols = [col for col in df.columns if col != label_col]  # 将特征列提取出来
    split_thresholds = {}  # 创建分箱阈值空字典

    for col in feature_cols:  # 遍历特征列中的每一列
        feature = df[col]  # 提取出特征列中正在遍历的这一列
        unique_count = feature.nunique()  # 得到该特征列中唯一值的数量

        if unique_count <= 5:  # 特征列中唯一值的数量不超过5，认为是离散值，不用分箱
            discrete_df[col] = feature.astype(
                str
            )  # 将特征转变为字符串，避免被误认为连续值
            split_thresholds[col] = None  # 离散特征无分箱阈值，记录为None
        else:  # 特征列中的唯一值数量超过5，认为是连续值，需要分箱
            if method == "quantile":  # 采用等频分箱的方式
                binned, bin_edges = (
                    pd.qcut(  # pd.qcut返回两个值：分箱后的区间标签（binned）、分箱边界（bin_edges）
                        feature,  # 传入需要分箱的特征列
                        q=bins,  # 指定分箱数
                        retbins=True,  # 返回分箱边界
                        duplicates="drop",  # 处理重复边界
                    )
                )
                thresholds = bin_edges[1:-1]  # 提取分箱阈值
            elif method == "uniform":  # 采用等距分箱的方式
                binned, bin_edges = (
                    pd.cut(  # pd.cut返回两个值：分箱后的区间标签（binned）、分箱边界（bin_edges）
                        feature,  # 传入需要分箱的特征列
                        bins=bins,  # 指定分箱数
                        retbins=True,  # 返回分箱边界
                        include_lowest=True,  # 确保最小值被包含在第一个区间，为处理左闭右开导致的边界问题
                        duplicates="drop",  # 处理重复边界
                    )
                )
                thresholds = bin_edges[1:-1]  # 提取分箱阈值
            else:  # 其他情况
                raise ValueError(f"不支持的分箱方法: {method}")  # 打印错误信息

            discrete_df[col] = binned.astype(
                str
            )  # 将分箱后的区间标签转换为字符串类型，便于后续模型处理
            split_thresholds[col] = (
                thresholds.tolist()
            )  # 记录当前特征的分箱阈值，转换为列表格式存储

    return discrete_df, split_thresholds  # 返回离散化后的数据和分箱阈值字典


# 方法一：等频分箱
discrete_df, _ = discretize_features(df, bins=3, method="quantile")

# 方法二：等距分箱
# discrete_df, _ = discretize_features(df, bins = 3, method = 'uniform')

### 数据集准备
构建机器学习模型前要构建训练和测试的数据集。在本例中首先需要分开标签和特征，标签是不能作为模型的输入特征的，就好比作业和试卷答案不能在做题和考试前就告诉学生。测试一个模型在一个任务上的效果至少需要训练集和测试集，训练集用来训练模型的参数，好比学生做作业获得知识，测试集用来测试模型效果，好比期末考试考察学生学习情况。测试集的样本不应该出现在训练集中，否则会造成模型效果估计偏高，好比考试时出的题如果是作业题中出现过的，会造成考试分数不能准确衡量学生的学习情况，估计值偏高。划分训练集和测试集有多种方法，下面首先介绍的是随机取一部分如20%作测试集，剩下作训练集。sklearn提供了相关工具函数`train_test_split`。sklearn的输入输出一般为numpy的array矩阵，需要先将pandas的DataFrame取出为numpy的array矩阵。

In [ ]:
all_y = discrete_df["blueWins"].values  # 所有标签数据
feature_names = discrete_df.columns[1:]  # 所有特征的名称
all_x = discrete_df[
    feature_names
].values  # 所有原始特征值，pandas的DataFrame.values取出为numpy的array矩阵

# 划分训练集和测试集
x_train, x_test, y_train, y_test = train_test_split(
    all_x, all_y, test_size=0.2, random_state=RANDOM_SEED
)
print(
    all_x.shape, all_y.shape, x_train.shape, x_test.shape, y_train.shape, y_test.shape
)  # 输出数据行列信息

print("========================================================")

# 划分训练集和验证集
x_train, x_validation, y_train, y_validation = train_test_split(
    x_train, y_train, test_size=0.2, random_state=RANDOM_SEED
)
print(x_train.shape, x_validation.shape, y_train.shape, y_validation.shape)

###  决策树模型的实现
***本小节要求实现决策树模型，请补全算法代码***

In [ ]:
class DecisionTree:
    def __init__(
        self,
        classes,
        features,
        max_depth=5,
        min_samples_split=100,
        min_impurity_decrease=-1,
        criterion="entropy",
    ):
        """
        初始化决策树模型

        :param classes: 类别列表
        :param features: 特征名称列表
        :param max_depth: 树的最大深度，预剪枝参数，防止树过深导致过拟合，默认为5
        :param min_samples_split: 结点分裂的最小样本数，预剪枝参数，样本数低于此值不再分裂，默认为100
        :param min_impurity_decrease: 基于混杂度下降的阈值，预剪枝参数，默认为-1，代表不使用
        :param criterion: 混杂度计算方式（'entropy'或'gini'），使用信息熵还是基尼系数，默认为信息熵
        """

        # 存储分类相关参数
        self.classes = classes  # 存储所有可能的分类类别
        self.features = features  # 存储特征名称列表

        # 存储剪枝相关参数
        self.max_depth = max_depth  # 预剪枝：树的最大深度限制
        self.min_samples_split = min_samples_split  # 预剪枝：结点分裂所需最小样本数
        self.min_impurity_decrease = min_impurity_decrease  # 预剪枝：基于信息增益的阈值

        # 存储混杂度计算类型
        self.criterion = criterion  # 混杂度计算方式

        # 树结构初始化
        self.root = None  # 决策树根结点（初始化为None，训练后填充）

    def _impurity(self, labels):
        """
        计算结点的混杂度

        :param: labels 结点样本标签列表
        :return: 混杂度
        """

        # 处理空结点情况，防止出现除以0错误
        if len(labels) == 0:  # 空结点混杂度为0
            return 0.0  # 无数据时无需计算，直接返回0.0

        # 统计每个类别出现的次数
        class_counts = defaultdict(int)  # 使用默认字典存储类别计数
        for label in labels:  # 遍历所有样本标签
            class_counts[label] += 1  # 对应类别计数+1

        # 计算每个类别占总样本的比例
        total = len(labels)  # 总样本数
        probs = np.array(
            [count / total for count in class_counts.values()]
        )  # 计算概率数组

        # 根据衡量混杂度的类型选择计算方式
        if self.criterion == "entropy":
            # 熵计算公式：-Σ(p*log2(p))，其中p是各分类的概率
            # log2(p)当p=0时无定义，但由于probs来自实际存在的类别，所以p>0
            return -np.sum(probs * np.log2(probs))  # 返回信息熵值

        elif self.criterion == "gini":
            # 基尼系数计算公式：1-Σ(p²)，反映随机选两个样本类别不同的概率
            return 1 - np.sum(np.power(probs, 2))  # 返回基尼系数值

        else:
            # 异常处理：输入错误，不支持的混杂度类型
            raise ValueError(f"不支持的混杂度类型: {self.criterion}")  # 抛出错误提示

    def _gain(self, X, y, feature_idx):
        """
        计算特征的混杂度下降量（父结点混杂度 - 子结点加权平均混杂度）

        :param X: 特征矩阵（n_samples × n_features），当前结点的样本特征
        :param y: 标签数组（n_labels），当前结点的样本标签
        :param feature_idx: 待计算信息增益的特征索引
        :return: 混杂度下降量
        """

        # 计算父结点的混杂度
        parent_impurity = self._impurity(y)  # 使用_impurity方法计算父结点混杂度
        total_samples = len(y)  # 父结点总样本数
        child_impurity = 0.0  # 初始化子结点混杂度加权和

        # 获取当前特征的所有唯一取值
        feature_values = np.unique(X[:, feature_idx])  # 提取该特征的所有不同值

        # 遍历每个特征值，计算对应子结点的混杂度
        for val in feature_values:
            # 筛选出特征值等于val的样本，生成布尔掩码
            mask = X[:, feature_idx] == val  # 布尔数组，True表示该样本特征值为val
            child_y = y[mask]  # 子结点的标签数组，仅保留mask为True的样本
            child_size = len(child_y)  # 子结点样本数

            if child_size == 0:  # 子结点无样本时跳过
                continue

            # 计算子结点的混杂度，并按样本比例加权求和
            # 子结点混杂度贡献 = Σ(子结点样本数/父结点样本数) * 子结点混杂度
            child_impurity += (child_size / total_samples) * self._impurity(child_y)

        # 混杂度下降量 = 父结点混杂度 - 子结点混杂度的加权平均值
        return parent_impurity - child_impurity  # 返回混杂度下降量

    def _majority_class(self, labels):
        """
        获取多数类别（用于叶结点分类，当无法分裂时选择出现次数最多的类别）

        :param labels: 样本标签列表
        :return: 多数类别标签，即出现次数最多的类别的标签
        """

        # 统计每个类别的出现次数
        class_counts = defaultdict(int)  # 默认字典存储类别计数
        for label in labels:  # 遍历所有标签
            class_counts[label] += 1  # 对应类别计数+1

        # 返回出现次数最多的类别
        return max(class_counts, key=lambda k: class_counts[k])  # 返回多数类别

    def _expand_node(self, X, y, depth, used_features):
        """
        递归扩展决策树结点（决策树的核心构建逻辑）

        :param X: 特征矩阵（当前结点的样本特征）
        :param y: 标签数组（当前结点的样本标签）
        :param depth: 当前结点深度（根结点深度为1）
        :param used_features: 已使用的特征索引列表（防止同一特征重复分裂）
        :return: 结点字典（叶结点或内部结点）
        """

        # 停止条件1：所有样本属于同一类别（无需划分）
        if len(np.unique(y)) == 1:  # 检查标签的唯一值数量
            return {"type": "leaf", "class": y[0]}  # 返回叶结点，类别为唯一的标签

        # 停止条件2：达到最大深度限制（预剪枝）
        if depth > self.max_depth:  # 当前深度超过设定的最大深度
            return {
                "type": "leaf",
                "class": self._majority_class(y),
            }  # 返回叶结点（类型为多数类别）

        # 停止条件3：样本数小于最小分裂数（预剪枝）
        if len(y) < self.min_samples_split:  # 当前结点样本数不足分裂要求
            return {
                "type": "leaf",
                "class": self._majority_class(y),
            }  # 返回叶结点（类型为多数类别）

        # 停止条件4：无可用特征（所有特征已分裂过，无法划分）
        length_features = len(self.features)  # 统计所有特征的数量
        used_features_set = set(used_features)  # 将已使用过的特征转换为集合
        available_features = [
            i for i in range(length_features) if i not in used_features_set
        ]  # 可用特征索引
        if not available_features:  # 没有可用特征时
            return {
                "type": "leaf",
                "class": self._majority_class(y),
            }  # 返回叶结点（类型为多数类别）

        # 寻找信息增益最大的特征（选择最佳分裂特征）
        best_gain = -np.inf  # 初始化最佳信息增益为负无穷
        best_feature = -1  # 初始化最佳特征索引

        for feature_idx in available_features:  # 遍历所有可用特征
            current_gain = self._gain(X, y, feature_idx)  # 计算当前特征的信息增益
            if current_gain > best_gain:  # 如果当前增益更大
                best_gain = current_gain  # 更新最佳增益
                best_feature = feature_idx  # 更新最佳特征索引

        # 停止条件5：混杂度下降量不足（预剪枝）
        # 仅当min_impurity_decrease非负且信息增益小于等于阈值时，返回叶节点
        if (
            self.min_impurity_decrease >= 0 and best_gain <= self.min_impurity_decrease
        ):  # 满足信息增益小于等于阈值
            return {
                "type": "leaf",
                "class": self._majority_class(y),
            }  # 返回叶结点（类型为多数类别）

        # 创建内部结点并分裂，选择最佳特征后生成子结点
        feature_values = np.unique(X[:, best_feature])  # 获取最佳特征的所有唯一值
        children = {}  # 存储子结点的字典（特征值: 子结点）
        current_majority = self._majority_class(
            y
        )  # 当前结点的多数类别（能够处理未知特征）

        for val in feature_values:  # 遍历最佳特征的每个取值
            mask = X[:, best_feature] == val  # 生成该特征值的样本掩码
            child_X = X[mask]  # 子结点的特征矩阵（仅保留mask为True的样本）
            child_y = y[mask]  # 子结点的标签数组（仅保留mask为True的标签）

            if len(child_y) == 0:  # 子结点无样本
                children[val] = {
                    "type": "leaf",
                    "class": current_majority,
                }  # 子结点用当前结点的多数类别作为叶结点（处理边界情况）
            else:  # 递归创建子结点（深度+1，标记当前特征为已使用）
                children[val] = self._expand_node(
                    child_X,
                    child_y,
                    depth + 1,
                    used_features + [best_feature],  # 将当前特征加入已使用列表
                )

        # 返回内部结点字典（记录划分特征和子结点）
        return {
            "type": "internal",  # 结点类型：内部结点（非叶结点）
            "feature_idx": best_feature,  # 划分特征索引（对应self.features中的位置）
            "children": children,  # 子结点字典（键为特征值，值为子结点）
            "majority_class": current_majority,  # 当前结点多数类别（能够处理未知特征）
        }

    def _calculate_error(self, node, X_val, y_val):
        """
        计算结点在验证集上的预测误差（用于后剪枝）

        :param node: 待评估的决策树结点
        :param X_val: 验证集特征矩阵
        :param y_val: 验证集标签数组
        :return: 预测误差（错误样本数/总样本数）
        """

        # 临时修改根结点为当前node，便于计算预测误差
        original_root = self.root  # 保存原始根结点
        self.root = node  # 临时替换根结点为当前结点，便于预测
        predictions = self.predict(X_val)  # 对验证集进行预测
        self.root = original_root  # 恢复原始根结点

        # 计算错误率（预测错误样本数 / 总样本数）
        return np.sum(predictions != y_val) / len(y_val)  # 返回误差值

    def _prune(self, node, X_val, y_val):
        """
        递归后剪枝函数（通过验证集误差判断是否剪枝）

        :param node: 当前待剪枝的结点
        :param X_val: 验证集特征矩阵
        :param y_val: 验证集标签数组
        :return: 剪枝后的结点（可能是原结点或剪枝后的叶结点）
        """

        # 如果是叶结点，直接返回（无需剪枝）
        if node["type"] == "leaf":  # 叶结点无法再剪枝
            return node

        # 递归剪枝所有子结点（先剪枝子树，再处理当前结点）
        for val in node["children"]:  # 遍历当前结点的所有子结点
            # 递归调用_prune处理子结点（子结点可能被剪枝为叶结点）
            node["children"][val] = self._prune(node["children"][val], X_val, y_val)

        # 计算当前结点剪枝前的误差（使用当前完整子树预测）
        original_error = self._calculate_error(node, X_val, y_val)

        # 创建剪枝后的叶结点（将当前结点变为叶结点，使用多数类别）
        pruned_node = {
            "type": "leaf",  # 结点类型变为叶结点
            "class": node["majority_class"],  # 类别为当前结点的多数类别
        }

        # 计算剪枝后的误差
        pruned_error = self._calculate_error(pruned_node, X_val, y_val)

        # 比较误差：如果剪枝后误差更小或相等，则执行剪枝（用叶结点替换当前结点）
        if pruned_error <= original_error:
            return pruned_node  # 返回剪枝后的叶结点
        else:
            return node  # 保留原结点，不剪枝

    def fit(self, X, y, X_val=None, y_val=None):
        """
        训练决策树模型

        :param X: 训练集特征矩阵（n_samples × n_features）
        :param y: 训练集标签数组（n_samples）
        :param X_val: 验证集特征矩阵（可选，用于后剪枝）
        :param y_val: 验证集标签数组（可选，用于后剪枝）
        """

        # 输入校验：特征数量必须与初始化的features列表一致
        assert len(self.features) == X.shape[1], (
            "特征数量不匹配: 初始化features与输入X的列数不一致"
        )

        # 构建初始决策树（使用预剪枝参数：max_depth, min_samples_split, 可能有min_impurity_decrease）
        self.root = self._expand_node(
            X, y, depth=1, used_features=[]
        )  # 从根结点开始递归构建

        # 如果提供了验证集，则进行后剪枝
        if X_val is not None and y_val is not None:
            self.root = self._prune(
                self.root, X_val, y_val
            )  # 调用_prune方法从根结点开始递归剪枝整个树

    def _traverse_node(self, node, sample):
        """
        递归遍历决策树进行预测

        :param node: 当前遍历的结点（初始为根结点）
        :param sample: 单个样本特征数组（长度等于特征数）
        :return: 预测的类别标签
        """

        # 到达叶结点，返回存储的类别
        if node["type"] == "leaf":  # 叶结点是最终分类结点
            return node["class"]  # 返回叶结点存储的类别

        # 获取分裂特征的索引和当前样本的特征值
        feature_idx = node["feature_idx"]  # 分裂特征的索引（对应self.features中的位置）
        sample_val = sample[feature_idx]  # 当前样本在该特征上的值

        # 处理训练中未出现的特征值（使用当前结点的多数类别）
        if sample_val not in node["children"]:  # 样本特征值不在训练时的分裂值中
            return node["majority_class"]  # 返回当前结点的多数类别

        # 递归进入对应的子结点（根据样本的特征值选择子结点）
        return self._traverse_node(node["children"][sample_val], sample)

    def predict(self, X):
        """
        预测样本的类别标签

        :param X: 特征矩阵（一维或二维numpy数组）
                  一维：单个样本（shape=(n_features,)）
                  二维：多个样本（shape=(n_samples, n_features)）
        :return: 预测标签数组（一维数组）
        """

        # 输入维度校验（必须是1维或2维）
        assert len(X.shape) in (1, 2), "输入维度必须为1（单个样本）或2（多个样本）"

        # 处理单个样本的情况（转换为二维数组便于统一处理）
        if len(X.shape) == 1:  # 输入是一维数组（单个样本）
            return self._traverse_node(self.root, X)  # 直接遍历树返回预测类别
        else:  # 输入是二维数组（多个样本）
            return np.array(
                [self._traverse_node(self.root, sample) for sample in X]
            )  # 对每个样本独立预测，转换为numpy数组返回

### 模型测试

In [ ]:
# 定义决策树模型
DT = DecisionTree(  # 实例化决策树
    classes=[0, 1],  # 标签类别
    features=feature_names,  # 所有特征的名称
    max_depth=5,  # 决策树的最大深度
    min_samples_split=600,  # 结点分裂的最小样本数
    criterion="gini",  # 混杂度计算方式
)

# 训练模型
DT.fit(x_train, y_train, x_validation, y_validation)  # 使用验证集进行后剪枝

# 预测测试集
y_predict = DT.predict(x_test)

# 评估准确率
test_acc = accuracy_score(y_predict, y_test)  # 计算预测准确率
print("预测结果分布：", Counter(y_predict))  # 输出预测结果分布
print(f"准确率: {test_acc:.2%}")  # 输出保留两位小数的百分数形式

### 模型调优
第一次模型测试结果可能不够好，可以先检查调试代码是否有bug，再尝试调整参数或者优化计算方法。

### 总结
一个完整的机器学习任务包括：确定任务、数据分析、特征工程、数据集划分、模型设计、模型训练和效果测试、结果分析和调优等多个阶段，本案例以英雄联盟游戏胜负预测任务为例，给出了每个阶段的一些简单例子，帮助大家入门机器学习，希望大家有所收获！